# Track Labeling Tool

Use this notebook to **manually label takeoff and landing points** in IGC/GPX tracks.
Labeled data is saved as JSON test cases consumed by `src/run_tests.ts`.

### Quick-start
1. Install dependencies (first time only): `pip install ipywidgets ipyleaflet plotly numpy`
2. Run all cells (`Kernel → Restart & Run All`)
3. Set `TRACK_FILE` in the **Configuration** cell
4. Re-run the **Load** and **UI** cells
5. Drag the sliders to select takeoff/landing, then click **Save Test Case**


In [ ]:
import json, math, os, re, warnings
from datetime import datetime, timezone
from pathlib import Path
import xml.etree.ElementTree as ET

warnings.filterwarnings('ignore')

try:
    import ipywidgets as widgets
    import ipyleaflet as ipl
    import plotly.graph_objects as go
    from IPython.display import display
    print("All packages loaded.")
except ImportError as e:
    print(f"Missing: {e}")
    print("Run: pip install ipywidgets ipyleaflet plotly")
    raise


In [ ]:
# ---- Geometry ----

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2) ** 2
    return 2 * R * math.asin(math.sqrt(min(1.0, a)))


# ---- IGC parser ----

def parse_igc(filepath):
    with open(filepath, 'r', errors='replace') as fh:
        lines = fh.readlines()

    date_m = None
    for line in lines:
        date_m = re.search(r'HFDTE(?:DATE:)?(\d{2})(\d{2})(\d{2})', line)
        if date_m:
            break
    if not date_m:
        raise ValueError("No HFDTE record found in IGC file")

    dd, mm, yy = int(date_m.group(1)), int(date_m.group(2)), int(date_m.group(3))
    year = 2000 + yy if yy < 70 else 1900 + yy
    base_epoch = datetime(year, mm, dd, tzinfo=timezone.utc).timestamp()

    fixes = []
    prev_s, day_offset = -1, 0
    for line in lines:
        line = line.rstrip()
        if not line.startswith('B') or len(line) < 35:
            continue
        try:
            hh, mi, ss = int(line[1:3]), int(line[3:5]), int(line[5:7])
            total_s = hh * 3600 + mi * 60 + ss
            if prev_s >= 0 and total_s < prev_s - 3600:
                day_offset += 86400
            prev_s = total_s

            lat_d = int(line[7:9])
            lat = lat_d + (int(line[9:11]) + int(line[11:14]) / 1000.0) / 60.0
            if line[14] == 'S':
                lat = -lat
            lon_d = int(line[15:18])
            lon = lon_d + (int(line[18:20]) + int(line[20:23]) / 1000.0) / 60.0
            if line[23] == 'W':
                lon = -lon

            press_alt = int(line[25:30])
            gps_alt   = int(line[30:35])
            alt = press_alt if press_alt > 0 else gps_alt
            ts_ms = (base_epoch + day_offset + total_s) * 1000.0
            fixes.append({
                'timestamp': ts_ms,
                'time':      datetime.fromtimestamp(ts_ms / 1000, tz=timezone.utc).isoformat(),
                'latitude':  lat, 'longitude': lon,
                'valid':     line[24] == 'A',
                'altitude':  alt,
            })
        except (ValueError, IndexError):
            continue

    if not fixes:
        raise ValueError("No valid B records found in IGC file")
    return fixes


# ---- GPX parser ----

def parse_gpx(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()
    ns_m = re.match(r'\{(.+?)\}', root.tag)
    ns = ns_m.group(1) if ns_m else 'http://www.topografix.com/GPX/1/1'

    fixes = []
    for trkpt in root.iter(f'{{{ns}}}trkpt'):
        try:
            lat = float(trkpt.get('lat'))
            lon = float(trkpt.get('lon'))
            time_el = trkpt.find(f'{{{ns}}}time')
            if time_el is None:
                continue
            dt = datetime.fromisoformat(time_el.text.strip().replace('Z', '+00:00'))
            ele_el = trkpt.find(f'{{{ns}}}ele')
            alt = float(ele_el.text) if ele_el is not None else None
            fixes.append({
                'timestamp': dt.timestamp() * 1000.0,
                'time':      dt.isoformat(),
                'latitude':  lat, 'longitude': lon,
                'valid':     True,
                'altitude':  alt,
            })
        except (ValueError, TypeError):
            continue

    fixes.sort(key=lambda f: f['timestamp'])
    return fixes


# ---- Generic loader ----

def load_track(filepath):
    ext = Path(filepath).suffix.lower()
    if ext == '.gpx':
        return parse_gpx(filepath)
    elif ext == '.igc':
        return parse_igc(filepath)
    else:
        with open(filepath, 'r', errors='replace') as fh:
            head = fh.read(16).lstrip()
        return parse_gpx(filepath) if head.startswith('<') else parse_igc(filepath)


def filter_fixes(fixes):
    seen = set()
    out = []
    for f in fixes:
        if f.get('valid', True) and f['timestamp'] not in seen:
            seen.add(f['timestamp'])
            out.append(f)
    return out


## Configuration

Change `TRACK_FILE` to point to your IGC or GPX file.
Files live under `tests/tracks/` by convention.


In [ ]:
# ---- Change this path ----
TRACK_FILE = "tests/tracks/my_flight.igc"

# Where to write labeled test cases
TEST_CASES_DIR = Path("tests/labeled")
TEST_CASES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Track file : {TRACK_FILE}")
print(f"Output dir : {TEST_CASES_DIR.resolve()}")


In [ ]:
raw   = load_track(TRACK_FILE)
fixes = filter_fixes(raw)

if len(fixes) < 5:
    raise ValueError(f"Only {len(fixes)} valid fixes — is the file correct?")

dur_min = (fixes[-1]['timestamp'] - fixes[0]['timestamp']) / 60_000
print(f"Loaded   : {len(fixes)} valid fixes")
print(f"Start    : {fixes[0]['time']}")
print(f"End      : {fixes[-1]['time']}")
print(f"Duration : {dur_min:.1f} min")


In [ ]:
# ===================================================================
# Interactive labeling UI
# Run this cell to display the tool. Add one segment per flight.
# ===================================================================

n = len(fixes)
segments = []

# ---- Map ----
lats = [f['latitude']  for f in fixes]
lons = [f['longitude'] for f in fixes]
STRIDE = max(1, n // 500)
m = ipl.Map(center=(sum(lats)/n, sum(lons)/n), zoom=13, scroll_wheel_zoom=True)
m.layout.height = '360px'
m.add(ipl.Polyline(locations=list(zip(lats[::STRIDE], lons[::STRIDE])),
                   color='royalblue', weight=2, opacity=0.8))

# ---- Altitude chart ----
times_s = [(f['timestamp'] - fixes[0]['timestamp']) / 1000 for f in fixes]
alts    = [f.get('altitude') or 0 for f in fixes]

fig = go.FigureWidget()
fig.add_trace(go.Scatter(x=times_s, y=alts, name='Altitude',
                         line=dict(color='steelblue', width=1.5)))
fig.update_layout(
    height=300,
    margin=dict(l=55, r=15, t=30, b=40),
    xaxis_title='Seconds from track start',
    yaxis_title='Altitude (m)',
    showlegend=False,
)

def redraw_shapes():
    shapes = []
    for seg in segments:
        shapes.append(dict(type='line', xref='x', yref='paper',
                           x0=times_s[seg['launch']], x1=times_s[seg['launch']],
                           y0=0, y1=1, line=dict(color='green', width=2, dash='dash')))
        shapes.append(dict(type='line', xref='x', yref='paper',
                           x0=times_s[seg['landing']], x1=times_s[seg['landing']],
                           y0=0, y1=1, line=dict(color='red', width=2, dash='dash')))
    fig.layout.shapes = shapes

# ---- Segment list ----
segments_box = widgets.VBox([])

def rebuild_segments_ui():
    for i, seg in enumerate(segments):
        seg['index_lbl'].value = f"<b>Flight {i + 1}</b>"
    segments_box.children = tuple(seg['container'] for seg in segments)

def make_segment():
    seg = {'launch': 0, 'landing': n - 1}

    index_lbl   = widgets.HTML(value='')
    launch_lbl  = widgets.HTML(
        value=f"<b style='color:green'>Launch  idx=0</b>  {fixes[0]['time']}")
    landing_lbl = widgets.HTML(
        value=f"<b style='color:red'>Landing idx={n-1}</b>  {fixes[-1]['time']}")

    launch_slider = widgets.IntSlider(
        value=0, min=0, max=n-1, step=1, description='Launch:',
        layout=widgets.Layout(width='95%'),
        style={'description_width': '70px'}, continuous_update=True,
    )
    landing_slider = widgets.IntSlider(
        value=n-1, min=0, max=n-1, step=1, description='Landing:',
        layout=widgets.Layout(width='95%'),
        style={'description_width': '70px'}, continuous_update=True,
    )

    launch_marker = ipl.CircleMarker(
        location=(fixes[0]['latitude'], fixes[0]['longitude']),
        radius=8, color='green', fill_color='limegreen', fill_opacity=1.0,
    )
    landing_marker = ipl.CircleMarker(
        location=(fixes[-1]['latitude'], fixes[-1]['longitude']),
        radius=8, color='darkred', fill_color='red', fill_opacity=1.0,
    )
    m.add(launch_marker)
    m.add(landing_marker)

    remove_btn = widgets.Button(description='Remove', button_style='danger', icon='trash',
                                layout=widgets.Layout(width='100px'))

    seg.update({
        'index_lbl': index_lbl,
        'launch_lbl': launch_lbl,    'landing_lbl': landing_lbl,
        'launch_slider': launch_slider, 'landing_slider': landing_slider,
        'launch_marker': launch_marker, 'landing_marker': landing_marker,
        'container': widgets.VBox([
            widgets.HBox([index_lbl, widgets.HTML("&nbsp;&nbsp;"), remove_btn]),
            launch_lbl, launch_slider,
            widgets.HTML("<div style='height:4px'></div>"),
            landing_lbl, landing_slider,
            widgets.HTML("<hr style='margin:6px 0; border-color:#ddd'>"),
        ]),
    })

    def on_launch(change):
        idx = change['new']
        seg['launch'] = idx
        f = fixes[idx]
        launch_marker.location = (f['latitude'], f['longitude'])
        launch_lbl.value = f"<b style='color:green'>Launch  idx={idx}</b>  {f['time']}"
        redraw_shapes()

    def on_landing(change):
        idx = change['new']
        seg['landing'] = idx
        f = fixes[idx]
        landing_marker.location = (f['latitude'], f['longitude'])
        landing_lbl.value = f"<b style='color:red'>Landing idx={idx}</b>  {f['time']}"
        redraw_shapes()

    def on_remove(b):
        m.remove(launch_marker)
        m.remove(landing_marker)
        segments.remove(seg)
        rebuild_segments_ui()
        redraw_shapes()

    launch_slider.observe(on_launch,  names='value')
    landing_slider.observe(on_landing, names='value')
    remove_btn.on_click(on_remove)

    segments.append(seg)
    rebuild_segments_ui()
    redraw_shapes()

# ---- Controls ----
add_btn     = widgets.Button(description='+ Add Flight', button_style='info', icon='plus')
desc_box    = widgets.Text(value='', description='Notes:',
                           placeholder='Optional description',
                           layout=widgets.Layout(width='99%'),
                           style={'description_width': '70px'})
save_btn    = widgets.Button(description='Save Test Case', button_style='success', icon='save')
status_html = widgets.HTML(value='')

add_btn.on_click(lambda b: make_segment())

def on_save(b):
    for i, seg in enumerate(segments):
        if seg['launch'] >= seg['landing']:
            status_html.value = (
                f"<span style='color:red'>Error: Flight {i+1} launch must come before landing.</span>")
            return

    test_case = {
        "file": TRACK_FILE,
        "description": desc_box.value.strip(),
        "created_at": datetime.now(timezone.utc).isoformat(),
        "expected": [{
            "launch_time":          fixes[seg['launch']]['time'],
            "landing_time":         fixes[seg['landing']]['time'],
            "launch_timestamp_ms":  fixes[seg['launch']]['timestamp'],
            "landing_timestamp_ms": fixes[seg['landing']]['timestamp'],
            "launch_idx":           seg['launch'],
            "landing_idx":          seg['landing'],
        } for seg in segments],
    }

    stem = Path(TRACK_FILE).stem
    out_path = TEST_CASES_DIR / f"{stem}.json"
    out_path.write_text(json.dumps(test_case, indent=2))
    status_html.value = f"<span style='color:green'>Saved → {out_path}</span>"
    print(f"Saved: {out_path}")

save_btn.on_click(on_save)

# ---- Initial segment + layout ----
make_segment()

display(widgets.VBox([
    widgets.HTML("<h3 style='margin:4px 0'>Track Labeling Tool</h3>"),
    desc_box,
    widgets.HTML("<hr style='margin:6px 0'>"),
    segments_box,
    add_btn,
    widgets.HTML("<hr style='margin:6px 0'>"),
    m, fig,
    widgets.HBox([save_btn, widgets.HTML("&nbsp;&nbsp;"), status_html]),
], layout=widgets.Layout(width='100%')))
